# Part 6 of 6: Results Summary & Discussion

COMP7015 Group Project – Topic 1: Plant Disease Classification. This notebook collects the key tables of notebooks 01–05 in one place (Section 21) and holds the discussion (Section 22).

The notebooks run in order: `01_data` → `02_cnn_resnet` → `03_vit` → `04_test_evaluation` → `05_robustness` → `06_discussion`. The shared code lives in the `plantdisease/` package. Finished training runs and predictions are saved under `OUT_DIR/runs/` and reused, so this notebook can be re-run on its own once the earlier ones have run.

In [3]:
import os

import pandas as pd
from IPython.display import Markdown, display

from plantdisease.config import out_path

TABLES = [
    ("Data: class counts per split", "class_counts.csv"),
    ("Background alone (Section 5.1)", "background_only.csv"),
    ("CNN / ResNet18 selection with all seeds (Section 9.3)", "tuning_multi_seed.csv"),
    ("ViT strategies: validation and cost (Section 10.3)", "vit_validation.csv"),
    ("ViT strategies: test (Section 10.4)", "vit_strategies_test.csv"),
    (
        "ViT strategies: same-seed comparisons (Section 10.4)",
        "vit_strategy_comparison.csv",
    ),
    ("Final models on the test set (Section 11)", "test_results.csv"),
    ("Compute trade-offs (Section 12)", "compute_tradeoffs.csv"),
    ("Errors that every seed makes (Section 13.1)", "stable_errors.csv"),
    ("Robustness on the same test leaves (Section 14)", "robustness.csv"),
    ("ViT retrained on other inputs (Section 15)", "input_training_grid.csv"),
    ("Same leaf, other background (Section 16)", "background_swap_tests.csv"),
    ("Attention on the leaf (Sections 17, 18.4)", "leaf_attention_countermeasure.csv"),
    ("Countermeasure (Section 18.3)", "countermeasure_results.csv"),
    (
        "Countermeasure vs. baseline, seed by seed (Section 18.3)",
        "countermeasure_comparison.csv",
    ),
    ("Left-out series GH_HL (Section 19.1)", "series_shift.csv"),
    ("PlantDoc field photos (Section 19.2)", "field_photos.csv"),
    ("Confidence and abstention (Section 20)", "confidence_abstention.csv"),
]

## 21. Results Summary

Every table below is read from `OUT_DIR`, where notebooks 01–05 save them. A missing table means that its notebook has not run yet.

In [4]:
for title, file_name in TABLES:
    display(Markdown(f"**{title}** (`{file_name}`)"))
    path = out_path(file_name)
    if os.path.exists(path):
        display(pd.read_csv(path))
    else:
        display(Markdown("*not available yet*"))

**Data: class counts per split** (`class_counts.csv`)

,class_name,train,val,test,All,leaf ids
0,Tomato___healthy,699,152,148,999,250
1,Tomato___Early_blight,699,152,148,999,250
2,Tomato___Leaf_Mold,668,144,140,952,238
3,Tomato___Spider_mites Two-spotted_spider_mite,1172,252,252,1676,419
4,All,3238,700,688,4626,1157


**Background alone (Section 5.1)** (`background_only.csv`)

,Unnamed: 0,accuracy,macro-F1
0,four image corners only,0.881429,0.868017
1,always the largest class (reference),0.360000,0.132353


**CNN / ResNet18 selection with all seeds (Section 9.3)** (`tuning_multi_seed.csv`)

,run,family,val macro-F1 (seed 62),mean over seeds,mean ± sd
0,SE-CNN (full recipe),cnn,1.000000,0.994078,0.9941 ± 0.0103
1,SE-CNN constant lr,cnn,1.000000,0.996599,0.9966 ± 0.0030
2,SE-CNN lr=3e-4,cnn,0.997363,0.998242,0.9982 ± 0.0015
3,"ResNet18 full FT, constant lr",resnet18,1.000000,1.000000,1.0000 ± 0.0000
4,"ResNet18 two-stage + cosine, lr=1e-4",resnet18,1.000000,0.999436,0.9994 ± 0.0010


... (26 outputs left out)

**PlantDoc field photos (Section 19.2)** (`field_photos.csv`)

*not available yet*

**Confidence and abstention (Section 20)** (`confidence_abstention.csv`)

,model,variant,errors,accuracy (all photos answered),accuracy (95% answered),share of errors among the 5% least confident,ECE
0,SE-CNN lr=3e-4,original,6.333333,0.990795,0.999490,0.933333,0.009556
1,SE-CNN lr=3e-4,background removed,518.333333,0.246609,0.243119,0.045064,0.695763
2,SE-CNN lr=3e-4,grayscale,393.666667,0.427810,0.438838,0.067989,0.466359
3,SE-CNN lr=3e-4,darker (x0.6),316.333333,0.540213,0.549439,0.070143,0.375169
4,SE-CNN lr=3e-4,blur (sigma=2),406.333333,0.409399,0.413863,0.057847,0.526068
5,SE-CNN lr=3e-4,noise (sigma=20),384.333333,0.441376,0.461774,0.085080,0.468001
6,"ResNet18 full FT, constant lr",original,5.000000,0.992733,0.998981,0.850000,0.006440
7,"ResNet18 full FT, constant lr",background removed,179.333333,0.739341,0.755352,0.111433,0.113800
8,"ResNet18 full FT, constant lr",grayscale,189.000000,0.725291,0.740571,0.103393,0.154858
9,"ResNet18 full FT, constant lr",darker (x0.6),16.333333,0.976260,0.987768,0.511638,0.012964


## 22. Discussion

*To be written from the tables above after the full run (3 seeds, full epochs). The results of the earlier notebook `plant_disease_classification3_julia.ipynb` used different classes (bacterial spot, late blight) and do not carry over.* Points to cover:

### Data protocol (notebook 01)
- Class counts, leaf ids and the photos excluded or left out; how much the background alone predicts the class (Section 5.1).

### Models and tuning (notebooks 02, 03)
- Which CNN and ResNet18 recipes were selected, and which ablations matter compared with the seed-to-seed variation.
- ViT fine-tuning strategies: the accuracy gain from head only → partial → full fine-tuning against the extra training time, trainable parameters and peak memory; whether the ranking holds in every seed; the effect of augmentation.

### Clean test, compute and errors (notebook 04)
- Mean ± sd of the three final models; which classes are confused; errors that every seed makes.
- Accuracy against inference cost and training cost.

### Robustness (notebook 05)
- Drops under background removal, grayscale, lighting, blur and noise; how much of the grayscale and background-removed drop disappears when the ViT is retrained on that input (Section 15).
- Whether predictions follow the leaf or the background (Sections 16, 17), and whether the countermeasure helps without costing clean accuracy (Section 18).
- The left-out healthy series and PlantDoc field photos (Section 19), with their limits: one class in the series test, two spider-mite photos in PlantDoc.
- Confidence and abstention under these changes (Section 20).

### Limitations
- One photo series per class: disease and background, camera and light are confounded.
- Three seeds on one fixed split: the standard deviation covers training randomness, not the choice of test leaves.
- Background-removed photos come from PlantVillage's own segmentation, which also cuts away some leaf regions.